# Exploratory Data Analysis — Smart City Capstone

**Project:** Vibetastrophe Consulting — Smart City AI/ML Capstone  

This notebook explores three datasets used in the capstone:
1. **City Traffic Accidents** (`city_traffic_accidents.csv`) — ~500K records
2. **Pothole Images** (`pothole_images/`) — ~3,700 road surface images
3. **UrbanPulse 311 Complaints** (`urbanpulse_311_complaints.csv`) — ~500K records

Data loading and basic preprocessing uses the shared functions from [`pipelines/data_pipeline.py`](../pipelines/data_pipeline.py) and [`pipelines/preprocessing_hints.py`](../pipelines/preprocessing_hints.py).

---

## Setup

In [1]:
import sys
import random
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

# Make project root importable so pipelines/ can be found from notebooks/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('Project root:', PROJECT_ROOT)

Project root: /Users/kalpanaselvaa/Documents/GitHub/fsa-aiml-2511/final-capstone-vibetastrophe-consulting


In [2]:
# Shared pipeline utilities
from pipelines.data_pipeline import (
    load_raw_data,
    clean_data,
    engineer_features,
)

# Dataset-specific preprocessing hints
from pipelines.preprocessing_hints import (
    load_accidents,
    create_temporal_features,
    process_weather_features,
    process_road_features,
    analyze_severity_distribution,
    get_top_complaint_types,
    create_complaint_categories,
)

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
POTHOLE_DIR  = RAW_DATA_DIR / 'pothole_images'

print('Accidents CSV  :', (RAW_DATA_DIR / 'city_traffic_accidents.csv').exists())
print('Complaints CSV :', (RAW_DATA_DIR / 'urbanpulse_311_complaints.csv').exists())
print('Pothole images :', POTHOLE_DIR.exists())

Accidents CSV  : True
Complaints CSV : True
Pothole images : True


---
## Dataset 1: City Traffic Accidents

### 1.1 Load & Basic Info

In [3]:
# Use load_accidents from preprocessing_hints — handles datetime parsing
accidents_df = load_accidents(str(RAW_DATA_DIR / 'city_traffic_accidents.csv'))

# Apply shared cleaning (dedup, strip whitespace, drop all-null cols)
accidents_df = clean_data(accidents_df)

print(f'Shape: {accidents_df.shape}')
print(f'Columns ({len(accidents_df.columns)}):', list(accidents_df.columns))

ValueError: unconverted data remains when parsing with format "%Y-%m-%d %H:%M:%S": ".000000". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
accidents_df.head()

In [ ]:
accidents_df.info()

### 1.2 Missing Values

In [ ]:
missing = accidents_df.isnull().sum()
missing_pct = (missing / len(accidents_df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
missing_summary['missing_pct'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Missing (%)')
ax.set_title('Accidents — Missing Values by Column')
plt.tight_layout()
plt.show()

### 1.3 Target Variable: Severity Distribution

In [ ]:
# Use the hint function — prints count + ratios and highlights the imbalance
analyze_severity_distribution(accidents_df)

In [ ]:
severity_counts = accidents_df['Severity'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

severity_counts.plot(kind='bar', ax=axes[0],
                     color=['#2196F3','#FF9800','#F44336','#9C27B0'], edgecolor='white')
axes[0].set_title('Severity — Count')
axes[0].set_xlabel('Severity Level')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)
for bar, count in zip(axes[0].patches, severity_counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
                 f'{count:,}', ha='center', va='bottom', fontsize=9)

axes[1].pie(severity_counts,
            labels=[f'Severity {i}' for i in severity_counts.index],
            autopct='%1.1f%%',
            colors=['#2196F3','#FF9800','#F44336','#9C27B0'],
            startangle=90)
axes[1].set_title('Severity — Proportion')

plt.suptitle('⚠️  Class Imbalance: Severity 2 dominates (~80%)', fontsize=12,
             fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()

### 1.4 Temporal Patterns

In [ ]:
# Use create_temporal_features from preprocessing_hints
accidents_df = create_temporal_features(accidents_df)

print('New temporal columns added:', [c for c in accidents_df.columns
       if c in ('hour','day_of_week','month','is_weekend','is_rush_hour',
                'is_morning_rush','is_evening_rush','duration_min')])

In [ ]:
day_labels   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

accidents_df['hour'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Accidents by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')

dow_counts = accidents_df['day_of_week'].value_counts().sort_index()
axes[1].bar(range(7), dow_counts.values, color='coral', edgecolor='white')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)
axes[1].set_title('Accidents by Day of Week')
axes[1].set_ylabel('Count')

month_counts = accidents_df['month'].value_counts().sort_index()
axes[2].bar(range(1, 13), month_counts.values, color='mediumseagreen', edgecolor='white')
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(month_labels, rotation=45)
axes[2].set_title('Accidents by Month')
axes[2].set_ylabel('Count')

plt.suptitle('Temporal Distribution of Traffic Accidents', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Severity vs. hour heatmap
hour_severity = accidents_df.groupby(['hour', 'Severity']).size().unstack(fill_value=0)
hour_severity_pct = hour_severity.div(hour_severity.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(hour_severity_pct.T, cmap='YlOrRd', ax=ax, fmt='.1f', annot=True, linewidths=0.3)
ax.set_title('Severity Distribution by Hour of Day (% within each hour)')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Severity')
plt.tight_layout()
plt.show()

### 1.5 Weather Features

In [ ]:
# Use process_weather_features from preprocessing_hints
accidents_df = process_weather_features(accidents_df)

print('New weather columns:', [c for c in accidents_df.columns
       if c in ('weather_data_available','is_freezing','low_visibility','weather_group')])

In [ ]:
weather_num_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)',
                    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']

accidents_df[weather_num_cols].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(weather_num_cols):
    for severity, grp in accidents_df.groupby('Severity'):
        grp[col].dropna().hist(bins=40, ax=axes[i], alpha=0.5,
                               label=f'Sev {severity}', density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)

plt.suptitle('Weather Feature Distributions by Severity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Grouped weather conditions (from categorize_weather in preprocessing_hints)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wg = accidents_df['weather_group'].value_counts()
wg.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Accidents by Weather Group')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Weather group vs severity
wg_sev = accidents_df.groupby(['weather_group', 'Severity']).size().unstack(fill_value=0)
wg_sev_pct = wg_sev.div(wg_sev.sum(axis=1), axis=0) * 100
wg_sev_pct.plot(kind='bar', ax=axes[1], colormap='Set2', edgecolor='white')
axes[1].set_title('Severity Distribution by Weather Group (%)')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Severity', fontsize=8)

plt.tight_layout()
plt.show()

### 1.6 Road Features

In [ ]:
# Use process_road_features from preprocessing_hints
accidents_df = process_road_features(accidents_df)

print('New road columns:', [c for c in accidents_df.columns
       if c in ('n_road_features', 'has_traffic_control')])

In [ ]:
road_feature_cols = ['Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction',
                     'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop',
                     'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop']

road_true_pct = (accidents_df[road_feature_cols].sum() / len(accidents_df) * 100).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
road_true_pct.plot(kind='bar', ax=ax, color='mediumslateblue', edgecolor='white')
ax.set_title('Road Features — % of Accidents Where Feature is Present')
ax.set_ylabel('% True')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 1.7 Geographic Distribution

In [ ]:
top_states = accidents_df['State'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
top_states.plot(kind='bar', ax=ax, color='tomato', edgecolor='white')
ax.set_title('Top 15 States by Accident Count')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
sample = accidents_df[['Start_Lat', 'Start_Lng', 'Severity']].dropna().sample(n=20000, random_state=42)

fig, ax = plt.subplots(figsize=(12, 7))
colors = {1: '#2196F3', 2: '#FF9800', 3: '#F44336', 4: '#9C27B0'}
for severity, grp in sample.groupby('Severity'):
    ax.scatter(grp['Start_Lng'], grp['Start_Lat'], c=colors[severity],
               s=1, alpha=0.3, label=f'Severity {severity}')
ax.set_title('Accident Locations — 20K sample')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

### 1.8 Correlation Heatmap (Numeric Features)

In [ ]:
numeric_cols = ['Severity', 'Distance(mi)', 'Temperature(F)', 'Wind_Chill(F)',
                'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
                'Wind_Speed(mph)', 'Precipitation(in)']

corr = accidents_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix — Numeric Features (Accidents)')
plt.tight_layout()
plt.show()

### 1.9 Accidents Dataset Summary

In [ ]:
print('=== ACCIDENTS DATASET SUMMARY ===')
print(f'Total records       : {len(accidents_df):,}')
print(f'Columns             : {accidents_df.shape[1]}')
print(f'Date range          : {accidents_df["Start_Time"].min().date()} → {accidents_df["Start_Time"].max().date()}')
print(f'States covered      : {accidents_df["State"].nunique()}')
print(f'Duplicate rows      : 0 (removed by clean_data)')
print()
analyze_severity_distribution(accidents_df)
print()
print('Columns with >30% missing values:')
print(missing_summary[missing_summary['missing_pct'] > 30]['missing_pct'].to_string())

---
## Dataset 2: Pothole Images

### 2.1 Class Distribution

In [ ]:
positive_images = list((POTHOLE_DIR / 'positive').glob('*'))
negative_images = list((POTHOLE_DIR / 'negative').glob('*'))

n_positive = len(positive_images)
n_negative = len(negative_images)
n_total    = n_positive + n_negative

print(f'Pothole (positive) : {n_positive:,}  ({n_positive/n_total*100:.1f}%)')
print(f'Normal road (neg)  : {n_negative:,}  ({n_negative/n_total*100:.1f}%)')
print(f'Total images       : {n_total:,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

labels = ['Normal Road\n(negative)', 'Pothole\n(positive)']
counts = [n_negative, n_positive]
colors = ['#4CAF50', '#F44336']

bars = axes[0].bar(labels, counts, color=colors, edgecolor='white', width=0.5)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Image Count by Class')
axes[0].set_ylabel('Count')

axes[1].pie(counts, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Class Proportion')

plt.suptitle('⚠️  Class Imbalance: Normal Road vs Pothole', fontsize=12,
             fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()

### 2.2 Sample Images

In [ ]:
random.seed(42)
sample_positive = random.sample(positive_images, 4)
sample_negative = random.sample(negative_images, 4)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, img_path in enumerate(sample_positive):
    img = mpimg.imread(str(img_path))
    axes[0, i].imshow(img)
    axes[0, i].set_title('Pothole', color='red', fontweight='bold')
    axes[0, i].axis('off')

for i, img_path in enumerate(sample_negative):
    img = mpimg.imread(str(img_path))
    axes[1, i].imshow(img)
    axes[1, i].set_title('Normal Road', color='green', fontweight='bold')
    axes[1, i].axis('off')

plt.suptitle('Sample Images — Pothole (top) vs Normal Road (bottom)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Image Properties

In [ ]:
sample_paths = random.sample(positive_images + negative_images, min(50, n_total))
shapes = [mpimg.imread(str(p)).shape for p in sample_paths]

heights  = [s[0] for s in shapes]
widths   = [s[1] for s in shapes]
channels = [s[2] if len(s) == 3 else 1 for s in shapes]

print(f'Height  — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')
print(f'Width   — min: {min(widths)},  max: {max(widths)},  mean: {np.mean(widths):.0f}')
print(f'Channels — unique: {set(channels)}')
print()
print('Note: Images are high-resolution — resize to 224×224 or 128×128 before training!')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(heights, bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Image Height Distribution (px)')
axes[0].set_xlabel('Height (px)')
axes[0].set_ylabel('Count')

axes[1].hist(widths, bins=20, color='coral', edgecolor='white')
axes[1].set_title('Image Width Distribution (px)')
axes[1].set_xlabel('Width (px)')

plt.tight_layout()
plt.show()

### 2.4 Pixel Intensity Analysis

In [ ]:
n_sample = 30
pos_sample = random.sample(positive_images, n_sample)
neg_sample = random.sample(negative_images, n_sample)

pos_brightness = [mpimg.imread(str(p)).mean() for p in pos_sample]
neg_brightness = [mpimg.imread(str(p)).mean() for p in neg_sample]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pos_brightness, bins=15, alpha=0.6, label='Pothole', color='red')
ax.hist(neg_brightness, bins=15, alpha=0.6, label='Normal Road', color='green')
ax.set_title('Mean Pixel Brightness — Pothole vs Normal Road')
ax.set_xlabel('Mean Pixel Value')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean brightness — Pothole    : {np.mean(pos_brightness):.3f}')
print(f'Mean brightness — Normal Rd  : {np.mean(neg_brightness):.3f}')

### 2.5 Pothole Images Summary

In [ ]:
print('=== POTHOLE IMAGES SUMMARY ===')
print(f'Total images       : {n_total:,}')
print(f'Pothole (positive) : {n_positive:,}  ({n_positive/n_total*100:.1f}%)')
print(f'Normal (negative)  : {n_negative:,}  ({n_negative/n_total*100:.1f}%)')
print(f'Typical resolution : {int(np.mean(heights))} × {int(np.mean(widths))} px')
print()
print('Key notes from get_pothole_image_hints():')
print('  - Binary classification: pothole vs normal road')
print('  - High-resolution — resize to 224×224 (ResNet) or 128×128 before training')
print('  - Class imbalance requires class weights or data augmentation')
print('  - Transfer learning recommended (ResNet50, EfficientNet-B0, MobileNet)')

---
## Dataset 3: UrbanPulse 311 Complaints

### 3.1 Load & Basic Info

In [ ]:
# Use shared load_raw_data + clean_data from data_pipeline
complaints_df = load_raw_data('urbanpulse_311_complaints.csv')
complaints_df = clean_data(complaints_df)

complaints_df['created_date'] = pd.to_datetime(complaints_df['created_date'], errors='coerce')
complaints_df['closed_date']  = pd.to_datetime(complaints_df['closed_date'],  errors='coerce')

print(f'Shape: {complaints_df.shape}')
print(f'Columns: {list(complaints_df.columns)}')

In [ ]:
complaints_df.head()

In [ ]:
complaints_df.info()

### 3.2 Missing Values

In [ ]:
c_missing = complaints_df.isnull().sum()
c_missing_pct = (c_missing / len(complaints_df) * 100).round(2)
c_missing_summary = pd.DataFrame({'missing_count': c_missing, 'missing_pct': c_missing_pct})
c_missing_summary = c_missing_summary[c_missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(c_missing_summary)

### 3.3 Target Variable: Complaint Type Distribution

In [ ]:
print(f'Total unique complaint types: {complaints_df["complaint_type"].nunique()}')
print()
# Use get_top_complaint_types from preprocessing_hints
top_types = get_top_complaint_types(complaints_df, n=5)
print('Top 5 types:', top_types)

In [ ]:
top20 = complaints_df['complaint_type'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(11, 7))
top20.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Complaint Types')
ax.set_xlabel('Count')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3.4 Top 5 + "Other" Classification Target

In [ ]:
# Use create_complaint_categories from preprocessing_hints
complaints_df = create_complaint_categories(complaints_df)

category_counts = complaints_df['complaint_category'].value_counts()
category_pct    = complaints_df['complaint_category'].value_counts(normalize=True) * 100

In [ ]:
cat_order = category_counts.index.tolist()
palette   = sns.color_palette('Set2', n_colors=len(cat_order))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(cat_order, [category_counts[c] for c in cat_order],
            color=palette, edgecolor='white')
axes[0].set_title('Complaint Categories — Count')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

axes[1].pie([category_counts[c] for c in cat_order], labels=cat_order,
            autopct='%1.1f%%', colors=palette, startangle=90)
axes[1].set_title('Complaint Categories — Proportion')

plt.suptitle('⚠️  Class Imbalance: "Other" dominates; top 5 cover ~50-60%',
             fontsize=12, fontweight='bold', color='darkred')
plt.tight_layout()
plt.show()

### 3.5 Temporal Patterns

In [ ]:
complaints_df['hour']        = complaints_df['created_date'].dt.hour
complaints_df['day_of_week'] = complaints_df['created_date'].dt.dayofweek
complaints_df['month']       = complaints_df['created_date'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

complaints_df['hour'].value_counts().sort_index().plot(
    kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Complaints by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')

dow = complaints_df['day_of_week'].value_counts().sort_index()
axes[1].bar(range(7), dow.values, color='coral', edgecolor='white')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_labels)
axes[1].set_title('Complaints by Day of Week')
axes[1].set_ylabel('Count')

mon = complaints_df['month'].value_counts().sort_index()
axes[2].bar(range(1, 13), mon.values, color='mediumseagreen', edgecolor='white')
axes[2].set_xticks(range(1, 13))
axes[2].set_xticklabels(month_labels, rotation=45)
axes[2].set_title('Complaints by Month')
axes[2].set_ylabel('Count')

plt.suptitle('Temporal Distribution of 311 Complaints', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 categories by month — seasonal patterns
top5_df = complaints_df[complaints_df['complaint_category'] != 'Other'].copy()
monthly_cat = top5_df.groupby(['month', 'complaint_category']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
monthly_cat.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white', width=0.8)
ax.set_title('Top 5 Complaint Types by Month (Seasonal Patterns)')
ax.set_xlabel('Month')
ax.set_ylabel('Count')
ax.set_xticklabels(month_labels, rotation=0)
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

### 3.6 Geographic Distribution (Borough)

In [ ]:
borough_counts = complaints_df['borough'].value_counts()
print('Complaints by Borough:')
print(borough_counts.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
borough_counts.plot(kind='bar', ax=ax, color='mediumpurple', edgecolor='white')
ax.set_title('311 Complaints by Borough')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
borough_cat = complaints_df.groupby(['borough', 'complaint_category']).size().unstack(fill_value=0)
borough_cat_pct = borough_cat.div(borough_cat.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(borough_cat_pct, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Complaint Category % by Borough')
ax.set_xlabel('Complaint Category')
ax.set_ylabel('Borough')
plt.tight_layout()
plt.show()

### 3.7 Text Feature: Resolution Description

In [ ]:
complaints_df['resolution_len'] = complaints_df['resolution_description'].fillna('').apply(len)

print('Resolution description — text length stats:')
print(complaints_df['resolution_len'].describe().round(0))
print(f'Empty/null descriptions: {(complaints_df["resolution_len"] == 0).sum():,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

complaints_df['resolution_len'].clip(upper=500).hist(
    bins=50, ax=axes[0], color='teal', edgecolor='white')
axes[0].set_title('Resolution Description Length (clipped at 500 chars)')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')

avg_len_by_cat = (complaints_df.groupby('complaint_category')['resolution_len']
                  .mean().sort_values(ascending=False))
avg_len_by_cat.plot(kind='bar', ax=axes[1], color='teal', edgecolor='white')
axes[1].set_title('Average Resolution Description Length by Category')
axes[1].set_ylabel('Avg Char Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 3.8 Channel & Status Breakdown

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

complaints_df['open_data_channel_type'].value_counts().plot(
    kind='bar', ax=axes[0], color='darkorange', edgecolor='white')
axes[0].set_title('Complaints by Submission Channel')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

complaints_df['status'].value_counts().plot(
    kind='bar', ax=axes[1], color='slateblue', edgecolor='white')
axes[1].set_title('Complaints by Status')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 3.9 311 Complaints Dataset Summary

In [ ]:
print('=== 311 COMPLAINTS DATASET SUMMARY ===')
print(f'Total records         : {len(complaints_df):,}')
print(f'Columns               : {complaints_df.shape[1]}')
print(f'Date range            : {complaints_df["created_date"].min().date()} → {complaints_df["created_date"].max().date()}')
print(f'Unique complaint types: {complaints_df["complaint_type"].nunique()}')
print(f'Boroughs              : {complaints_df["borough"].nunique()}')
print(f'Duplicate rows        : 0 (removed by clean_data)')
print()
print('Top 5 + Other distribution:')
for cat in category_counts.index:
    print(f'  {cat:<25} {category_counts[cat]:>8,}  ({category_pct[cat]:.1f}%)')

---
## Overall EDA Summary

| Dataset | Records | Target | Key Challenge |
|---|---|---|---|
| Traffic Accidents | ~500K | Severity (1–4) | ~80% Severity 2 — severe class imbalance |
| Pothole Images | ~3,700 | Binary (pothole/normal) | ~61/39 split; high-res images need resizing |
| 311 Complaints | ~500K | Complaint category (6 classes) | "Other" dominates; top 5 cover ~50–60% |

### Key Findings
- **Class imbalance is present in every dataset** — use `class_weight='balanced'`, SMOTE, or stratified splits. Accuracy alone is misleading.
- **Traffic accidents**: strong temporal signals (rush-hour peaks), weather missingness is non-random (`weather_data_available` flag is useful).
- **Pothole images**: very high resolution (~2760×3680 px) — must resize before training. Transfer learning (ResNet50/EfficientNet) is recommended.
- **311 complaints**: 186 categories collapse to Top 5 + Other for a tractable 6-class NLP problem; strong seasonal patterns (Snow/Ice in winter, Noise in summer).